# C1 backend 1 — OpenMask3D class-agnostic mask stage (mask module only)

**Scope:** raw `mesh.ply` → class-agnostic instance masks (`get_masks_single_scene.py`, NO SAM/CLIP feature stage) → frozen `SegmentationOutput` sidecar → one tar.gz on Drive.

**Isolation (contract G2):** never upload `info_semantic.json` or `mesh_semantic.ply` to this runtime. Only `mesh.ply` + the three oracle-free repo files.

**Drive layout:** `MyDrive/c1/<scene>/mesh.ply`, `MyDrive/c1/segmenter/{base,ply,mask_resolve}.py`, outputs → `MyDrive/c1/out/`.

**Runtime:** GPU (L4/A100 preferred; T4 may work for single rooms). Cells [6] is the known fight zone (upstream pins py3.10/CUDA 11.3/torch 1.12.1/MinkowskiEngine).

Local afterwards: `python3 tools/c1_run.py <room_dir> <bundle_dir> replica_<scene>`.

In [ ]:
# [1] Environment report
import subprocess, platform
print(platform.python_version(), platform.platform())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# [2] Frozen run config (all of this lands in meta.json)
SCENE = 'arkitscenes_41069021'   # ARKitScenes dev scene; see
# docs/arkitscenes_mask3d_contract.md. For the sealed transfer, run the
# whole notebook once as arkitscenes_41069025 and once as
# arkitscenes_41069042. Output paths contain SCENE, so the bundles cannot
# overwrite one another. Do not evaluate until the local pair check passes.
# Replica order was
# office_0 -> room_2 -> room_1 -> frl_apartment_0 (frl LAST).
# A wrong SCENE cannot run silently: cell [4] hard-gates on the mesh sha256.

OM3D_REPO = 'https://github.com/OpenMask3D/openmask3d'
OM3D_COMMIT = '3bc3fc52693b25668d0e91d55a2ea714544a4749'   # main @ 2023-12-15
# class-agnostic mask checkpoint for ARBITRARY scenes (OpenMask3D README):
CKPT_GDRIVE_ID = '1rD2Uvbsi89X4lSkont_jUTT7X9iaox9y'

# upstream single-scene defaults (compute_masks_single_scene.sh) — do not tune
NUM_QUERIES = 150
USE_DBSCAN = 'true'
DBSCAN_EPS = 0.95

# mask -> dense assignment (frozen rule, segmenter/mask_resolve.py). The
# dev artifact remains immutable at 0.4; sealed transfer runs deliver the
# declared ms02 operating point directly instead of needing local repair.
SEALED_TRANSFER_SCENES = {'arkitscenes_41069025', 'arkitscenes_41069042'}
MIN_SCORE = 0.2 if SCENE in SEALED_TRANSFER_SCENES else 0.4
MIN_VERTICES = 20

# pinned inputs (tools/replica_scenes.lock.json) — hard gates
MESH_SHA256 = {
    'room_1':          '21695deccc1fe76051d90178eccc1609ee1bab8b5dc715683dd17f7903cf6ee0',
    'room_2':          'e58a7c717c7922e1300ba20ae8053c5dbfdf9bd5f2515e10c71edad98bcb7e44',
    'office_0':        'cdb6ede0b9d455f491ef8fd63cd916a86a505b777842aabd6aa428edf9ff9032',
    'frl_apartment_0': '459374364b1fb6d61b28809fb2ebb722366ffc055caf990a5d659b1ebdd3e71b',
    # ARKitScenes canonical mesh, written by adapters/arkitscenes.py.
    # Gravity-aligned (up = +z) like ScanNet; same vertex order as the
    # P1 bank and the box oracle, which is what lets the banks pool.
    'arkitscenes_41069021': 'ec219f56c1f9d79a17f4ba0a224d19f75188aa38accf6a4074283d5f66c70d0b',
    'arkitscenes_41069025': '361ce587a7af33c1247db5eb6b1a56f6188a94202281a49f880812fada7b8770',
    'arkitscenes_41069042': 'fe2dc97c20d8566a9caded784388f635a5da997c5a6e713864c7f1f85c0ef661',
}
N_VERTICES = {'room_1': 645512, 'room_2': 722496, 'office_0': 589517,
              'frl_apartment_0': 1757500, 'arkitscenes_41069021': 1008964,
              'arkitscenes_41069025': 1064216,
              'arkitscenes_41069042': 422763}
DEVIATIONS = [
    'trainer.py patched to also save pred_scores (upstream saves only _masks.pt)',
    'model runs in a dedicated python 3.10 conda env (m3d); Colab kernel untouched',
    'nvcc 11.3 from nvidia/label/cuda-11.3.1 conda channel (subset; libnpp CDN artifact is corrupt)',
    'pip<24.1 + setuptools==69.5.1 (2022 pins have pre-PEP-508 metadata; torch 1.12 needs pkg_resources)',
    'built with gcc-9 (torch CUDA-11.3 gate requires g++ <= 10.0.0; Ubuntu g++-10 is 10.5)',
]

In [ ]:
# [3] Drive -> VM disk (single copies; no repeated small reads through the mount)
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
WORK = pathlib.Path('/content/c1'); (WORK / 'segmenter').mkdir(parents=True, exist_ok=True)
shutil.copy(f'/content/drive/MyDrive/c1/{SCENE}/mesh.ply', WORK / 'mesh.ply')
for f in ('base.py', 'ply.py', 'mask_resolve.py'):
    shutil.copy(f'/content/drive/MyDrive/c1/segmenter/{f}', WORK / 'segmenter' / f)
(WORK / 'segmenter' / '__init__.py').write_text('')
import sys; sys.path.insert(0, str(WORK))

In [ ]:
# [4] Hard gate: pinned mesh hash + vertex count
from segmenter.base import sha256_file
from segmenter.ply import parse_vertices
sha = sha256_file(WORK / 'mesh.ply')
assert sha == MESH_SHA256[SCENE], f'mesh hash mismatch: {sha}'
xyz = parse_vertices(WORK / 'mesh.ply')
assert len(xyz) == N_VERTICES[SCENE], f'{len(xyz)} != {N_VERTICES[SCENE]}'
print(f'{SCENE}: {len(xyz)} vertices, sha256 {sha[:16]}... OK')

In [ ]:
# [5] Clone pinned commit + patch trainer to ALSO save scores
# (single-scene path torch.saves only <scene>_masks.pt; pred_scores exists
#  in self.preds but never reaches disk — trainer.py ~line 723)
!git clone {OM3D_REPO} /content/openmask3d
!cd /content/openmask3d && git checkout {OM3D_COMMIT}
trainer = pathlib.Path('/content/openmask3d/openmask3d/class_agnostic_mask_computation/trainer/trainer.py')
src = trainer.read_text()
anchor = 'torch.save(self.preds[file_names[bid]][\'pred_masks\'].astype(np.float16), os.path.join(pred_save_folder, file_names[bid]+"_masks.pt"))'
assert anchor in src, 'trainer.py drifted from the pinned commit — re-check the patch anchor'
patch = anchor + '\n            np.save(os.path.join(pred_save_folder, file_names[bid]+"_scores.npy"), self.preds[file_names[bid]][\'pred_scores\'])'
trainer.write_text(src.replace(anchor, patch, 1))
print('patched: scores now saved alongside masks')

In [ ]:
# [6] Environment — kernel stays UNTOUCHED (no condacolab, no restart).
# Miniforge, not Miniconda: Anaconda's pkgs/main + pkgs/r channels are
# ToS-gated. CUDA: install only the compile-relevant 11.3 subset — the
# cuda-toolkit meta pulls libnpp, whose CDN artifact fails md5 (stale
# legacy-channel package), and nothing here needs npp/nvvp/samples/gdb.
# Subset determined empirically: nvcc + cudart + thrust/nvrtc + the libs
# whose headers torch/ME include (cublas, cusparse, curand, cusolver, nvtx).
%%bash
set -e
rm -rf /opt/conda   # clear any half-finished prior install
wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
bash /tmp/mf.sh -b -p /opt/conda
/opt/conda/bin/conda create -y -q -n m3d -c conda-forge --override-channels python=3.10
/opt/conda/bin/conda install -y -q -n m3d -c "nvidia/label/cuda-11.3.1" --override-channels \
  cuda-nvcc cuda-cudart cuda-thrust cuda-nvrtc libcublas libcusparse libcurand libcusolver cuda-nvtx
/opt/conda/envs/m3d/bin/python --version
/opt/conda/envs/m3d/bin/nvcc --version | tail -1

In [ ]:
# [6b] Model-env installs (upstream install_requirements.sh minus SAM/CLIP).
# All through the m3d env's pip; kernel never touched. Period-piece needs:
# (1) pip<24.1 — 2022 pins carry pre-PEP-508 metadata modern pip rejects;
# (2) setuptools<81 — torch 1.12 imports pkg_resources (removed in 81);
# (3) --no-build-isolation wherever setup.py imports torch;
# (4) gcc-9 — torch's CUDA-11.3 gate demands g++ <= 10.0.0 (Ubuntu g++-10
#     is 10.5.0, one minor too new);
# (5) ninja on PATH — without it torch falls back to a legacy compile path
#     with a known ME bug (host -fopenmp reaches nvcc: "Unknown option");
# (6) PIP_CONSTRAINT — 2026-08-08. pytorch-lightning 1.7.2 requires
#     torchmetrics>=0.7.0 UNPINNED at this point (the ==0.11.0 pin is nine
#     lines further down, far too late). pip resolves that to a current
#     torchmetrics, which requires torch>=2.0, and silently UPGRADES torch
#     off the pin. Nothing then fails until detectron2 / pointnet2 /
#     MinkowskiEngine compile against the wrong torch ~20 min later. The
#     tell is a block of 200-500 MB nvidia-*-cu12 wheels: torch 1.12.1+cu113
#     bundles CUDA statically and never pulls those. A constraints file
#     applies to EVERY pip invocation in this cell, so a package that
#     demands a newer torch now backtracks or fails loudly in place.
import os
PIP = '/opt/conda/envs/m3d/bin/pip'
PY  = '/opt/conda/envs/m3d/bin/python'
CONSTRAINTS = '/content/c1_torch_constraints.txt'
open(CONSTRAINTS, 'w').write('torch==1.12.1+cu113\ntorchvision==0.13.1+cu113\n')
os.environ['PIP_CONSTRAINT'] = CONSTRAINTS   # inherited by every ! below
# Fails at the step that moved torch, not four steps later. A `!` line does
# NOT halt the cell on a nonzero exit, so this has to be a Python assert.
def check_torch():
    import subprocess
    v = subprocess.run([PY, '-c', 'import torch;print(torch.__version__)'],
                       capture_output=True, text=True).stdout.strip()
    print('  torch ->', v or '(import failed)')
    assert v.startswith('1.12.1'), (
        f'torch drifted to {v!r}; the preceding install pulled a package '
        f'that requires a newer torch. See note (6) above.')
E = 'PATH=/opt/conda/envs/m3d/bin:$PATH CC=gcc-9 CXX=g++-9 CUDAHOSTCXX=g++-9 CUDA_HOME=/opt/conda/envs/m3d MAX_JOBS=4'
%cd /content/openmask3d
!apt-get -qq install -y libopenblas-dev gcc-9 g++-9 ninja-build > /dev/null
!{PIP} -q install 'pip<24.1'
!{PIP} -q install 'setuptools==69.5.1' wheel ninja==1.10.2.3
!{PIP} -q install torch==1.12.1+cu113 torchvision==0.13.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
check_torch()
# torchmetrics pinned HERE, alongside the package that pulls it, not later.
!{PIP} -q install pytorch-lightning==1.7.2 torchmetrics==0.11.0 fire imageio tqdm
check_torch()
!{PIP} -q install python-dotenv pyviz3d scipy plyfile scikit-learn trimesh loguru albumentations volumentations
check_torch()
!{PIP} -q install antlr4-python3-runtime==4.8 omegaconf==2.0.6 hydra-core==1.0.5 --no-deps
!{E} {PIP} -q install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git@710e7795d0eeadf9def0e7ef957eea13532e34cf'
# MinkowskiEngine compiles from source (~10-20 min); env nvcc 11.3; MAX_JOBS caps RAM
!{E} {PIP} install -v --no-build-isolation --no-deps -U git+https://github.com/NVIDIA/MinkowskiEngine 2>&1 | tail -25
!{PIP} install --no-build-isolation torch-scatter==2.1.0 -f https://data.pyg.org/whl/torch-1.12.1+cu113.html
!{PIP} -q install open3d==0.16.0 pycocotools h5py transforms3d fvcore cloudpickle 'Pillow==9.3.0' gorilla-core==0.2.7.8
check_torch()
!cd openmask3d/class_agnostic_mask_computation/third_party/pointnet2 && {E} {PIP} install --no-build-isolation .
# NOTE: CLIP / segment-anything deliberately NOT installed (feature stage skipped)
!{PY} -c "import torch, MinkowskiEngine as ME, pytorch_lightning, detectron2; print('torch', torch.__version__, '| ME', ME.__version__, '| pl', pytorch_lightning.__version__, '| cuda_ok', torch.cuda.is_available())"

In [ ]:
# [7] Checkpoint (arbitrary-scenes mask module) + pin its hash
!pip install -q gdown && gdown {CKPT_GDRIVE_ID} -O /content/mask_module.ckpt
from segmenter.base import sha256_file
CKPT_SHA = sha256_file(pathlib.Path('/content/mask_module.ckpt'))
print('checkpoint sha256:', CKPT_SHA)

In [ ]:
# [8]+[9] Mask-stage inference via the m3d env's python (mirrors upstream
# compute_masks_single_scene.sh). get_masks_single_scene.py reads the ply
# with open3d read_point_cloud — vertex COUNT and ORDER preserved, which the
# sidecar contract relies on. Output: <name>_masks.pt (num_points, num_masks)
# float16 0/1, sorted by score desc, + our patched <name>_scores.npy.
import os, time
MASK_DIR = '/content/mask_out'
!rm -rf {MASK_DIR}   # stale-mask guard: outputs are named mesh_masks.pt for
                     # EVERY scene, so leftovers from a prior scene on the
                     # same VM must never be picked up by [10]
t0 = time.time()
!cd /content/openmask3d/openmask3d && OMP_NUM_THREADS=3 {PY} \
  class_agnostic_mask_computation/get_masks_single_scene.py \
  general.experiment_name="single_scene" \
  general.checkpoint="/content/mask_module.ckpt" \
  general.train_mode=false \
  data.test_mode=test \
  model.num_queries={NUM_QUERIES} \
  general.use_dbscan={USE_DBSCAN} \
  general.dbscan_eps={DBSCAN_EPS} \
  general.save_visualizations=false \
  general.scene_path="/content/c1/mesh.ply" \
  general.mask_save_dir="{MASK_DIR}"
RUNTIME_S = time.time() - t0
print(f'inference wall time: {RUNTIME_S:.0f}s'); print(os.listdir(MASK_DIR))

In [ ]:
# [10] Deterministic resolution -> dense assignment (frozen, locally-tested
# rule). Runs on the KERNEL python (needs only numpy + torch-for-unpickling;
# weights_only=False because the .pt is a pickled numpy array, not weights).
import glob, numpy as np, torch
masks_pt = glob.glob(f'{MASK_DIR}/*_masks.pt')[0]
scores_npy = glob.glob(f'{MASK_DIR}/*_scores.npy')[0]
masks = torch.load(masks_pt, weights_only=False)   # (num_points, num_masks)
masks = np.asarray(masks, dtype=np.float16).T > 0  # -> [K, N] bool
scores = np.load(scores_npy).astype(float)
assert masks.shape[1] == N_VERTICES[SCENE], f'not full resolution: {masks.shape}'
assert masks.shape[0] == len(scores), f'{masks.shape[0]} masks vs {len(scores)} scores'
from segmenter.mask_resolve import MaskResolveConfig, resolve_masks
cfg = MaskResolveConfig(min_score=MIN_SCORE, min_vertices=MIN_VERTICES)
vertex_instance_ids = resolve_masks(masks, scores, cfg)
n_inst = len(np.unique(vertex_instance_ids[vertex_instance_ids >= 0]))
print(f'{masks.shape[0]} masks -> {n_inst} instances, '
      f'{float((vertex_instance_ids < 0).mean()):.1%} unclaimed')

In [ ]:
# [11]+[12] Export frozen sidecar + RAW masks, tar, copy the single archive
# to Drive. raw_masks.npz (packbits-compressed bool masks + scores) lets the
# local side re-resolve at different MIN_SCORE values without re-running the
# GPU — tools/c1_resolve_sweep.py / c1_reresolve.py consume it.
import json, subprocess
from segmenter.base import SegmentationOutput, save_segmentation_output
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
seg = SegmentationOutput(
    input_mesh_sha256=MESH_SHA256[SCENE], n_vertices=N_VERTICES[SCENE],
    segmenter_name='openmask3d_class_agnostic_mask3d',
    segmenter_version=OM3D_COMMIT[:12],
    config_params_json=json.dumps({
        'checkpoint_gdrive_id': CKPT_GDRIVE_ID, 'checkpoint_sha256': CKPT_SHA,
        'num_queries': NUM_QUERIES, 'use_dbscan': USE_DBSCAN, 'dbscan_eps': DBSCAN_EPS,
        **cfg.params(), 'deviations_from_upstream_env': DEVIATIONS,
    }, sort_keys=True),
    vertex_instance_ids=vertex_instance_ids,
    # instance ids are original mask rows -> confidence = that mask's score
    instance_confidence={int(i): float(scores[i])
                         for i in np.unique(vertex_instance_ids) if i >= 0},
    runtime_seconds=RUNTIME_S, hardware=gpu,
).finalize()
bundle = pathlib.Path(f'/content/c1/bundle_{SCENE}')
save_segmentation_output(seg, bundle)
np.savez_compressed(bundle / 'raw_masks.npz',
                    masks_packed=np.packbits(masks.astype(bool), axis=1),
                    n_vertices=np.int64(masks.shape[1]),
                    scores=scores)
!tar -czf /content/{SCENE}_c1_bundle.tar.gz -C /content/c1 bundle_{SCENE}
!mkdir -p /content/drive/MyDrive/c1/out
shutil.copy(f'/content/{SCENE}_c1_bundle.tar.gz', '/content/drive/MyDrive/c1/out/')
print('saved to Drive:', seg.output_sha256)

## [13] Local side (not in Colab)

```
tar -xzf <scene>_c1_bundle.tar.gz
python3 tools/c1_run.py /Users/deevyaswain/Desktop/datasets/replica/<scene> bundle_<scene> replica_<scene>
```

Verifies both hashes, runs the exact evaluator (C1.00), builds the oracle-enriched bundle (C1.03), prints B↔C1 battery diffs with per-segment attribution (C1.04).